In [ ]:
import os
import arcpy

#merge years 1994-2024

# Source GDB containing all the yearly layers
sefm_gdb = r"./data/SEFM_L_ABA_1994_2025_polys.gdb"

# Your working GDB where the merged output should go
working_gdb = r"./output/ClassiFIRE.gdb"

# Build list of full paths for 1994–2024
layers = [os.path.join(sefm_gdb, f"L_BurnedArea_{year}_poly")
          for year in range(1994, 2025)]

print(layers)

# Output path inside your working GDB
out_fc = os.path.join(working_gdb, "merged_94_24")

# Merge
arcpy.management.Merge(layers, out_fc)

print("Merged output saved to:", out_fc)

In [ ]:
import os
import arcpy

# Define working GDB relative to notebook directory
base_dir = os.path.dirname(os.path.abspath("__file__"))
working_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")

# Set feature class and frequency table paths inside the working GDB
fc = os.path.join(working_gdb, "merged_94_24")
freq_table = os.path.join(working_gdb, "nlcdr_frequencies")
# 1. Run Frequency tool (outputs a table with counts)
arcpy.analysis.Frequency(fc, freq_table, "nlcdr_domi")

# 2. Get total count
total = int(arcpy.management.GetCount(fc)[0])

# 3. Read the table and print percentages
print(f"Total detections: {total}")
with arcpy.da.SearchCursor(freq_table, ["nlcdr_domi", "FREQUENCY"]) as cursor:
    for row in cursor:
        cat = row[0]
        count = row[1]
        pct = (count / total) * 100
        print(f"{cat}: {count} detections ({pct:.3f}%)")

In [ ]:
#filter out detections where nlcdr_domi = 20 (developed, 2.9% of detections) or 11 (open water, 0.1%) 
import arcpy
import os

# Set input and output dataset paths inside the working GDB
in_fc = os.path.join(working_gdb, "merged_94_24")
out_fc = os.path.join(working_gdb, "merged_94_24_nlcd_filtered")

# SQL expression to EXCLUDE the two categories
sql = "nlcdr_domi NOT IN ('nlcdr_11', 'nlcdr_20')"

# Export the filtered subset
arcpy.management.MakeFeatureLayer(in_fc, "temp_lyr", sql)
arcpy.management.CopyFeatures("temp_lyr", out_fc)
arcpy.management.Delete("temp_lyr")

print("Filtered output saved to:", out_fc)

In [ ]:
#determine % of detections below 0.81 ha (2 acres) (mimimum detectable size reported by SEFM is 0.89 ha) (0% only 4 detections excluded)
import arcpy
import os

# Input (already NLCD-filtered) and Output (size-filtered) paths
in_fc = os.path.join(working_gdb, "merged_94_24_nlcd_filtered")
out_fc = os.path.join(working_gdb, "merged_94_24_nlcd_size_filtered")

# -------------------------------------------------------------------
# 1. Add area_ha field
# -------------------------------------------------------------------
fields = [f.name for f in arcpy.ListFields(in_fc)]
if "area_ha" not in fields:
    arcpy.management.AddField(in_fc, "area_ha", "DOUBLE")

# -------------------------------------------------------------------
# 2. Calculate area in hectares
# -------------------------------------------------------------------
arcpy.management.CalculateField(
    in_fc,
    "area_ha",
    "!shape.area@SQUAREMETERS! / 10000",
    "PYTHON3"
)

# -------------------------------------------------------------------
# 3. Calculate % of detections < 0.81 ha
# -------------------------------------------------------------------
# Total detections
total = int(arcpy.management.GetCount(in_fc)[0])

# Make a temporary layer for small detections
small_sql = "area_ha < 0.81"
arcpy.management.MakeFeatureLayer(in_fc, "small_lyr", small_sql)
small_count = int(arcpy.management.GetCount("small_lyr")[0])

pct_small = (small_count / total) * 100

print(f"Total detections: {total}")
print(f"Detections < 0.81 ha: {small_count} ({pct_small:.3f}%)")

# Clean up
arcpy.management.Delete("small_lyr")

# -------------------------------------------------------------------
# 4. Export detections >= 0.81 ha
# -------------------------------------------------------------------
keep_sql = "area_ha >= 0.81"

arcpy.management.MakeFeatureLayer(in_fc, "keep_lyr", keep_sql)
arcpy.management.CopyFeatures("keep_lyr", out_fc)
arcpy.management.Delete("keep_lyr")

print("Filtered output saved to:", out_fc)

In [ ]:
#assign each remaining detection a unique "detection_id" for tracking
import arcpy
import os

# Set dataset path relative to working GDB
fc = os.path.join(working_gdb, "merged_94_24_nlcd_size_filtered")

# 1. Add numeric detection_id field
fields = [f.name for f in arcpy.ListFields(fc)]
if "detection_id" not in fields:
    arcpy.management.AddField(fc, "detection_id", "LONG")

# 2. Populate detection_id with sequential integers
with arcpy.da.UpdateCursor(fc, ["detection_id"]) as cursor:
    counter = 1
    for row in cursor:
        row[0] = counter
        cursor.updateRow(row)
        counter += 1

print("Numeric detection_id assigned.")

In [ ]:
#we want to use prebd_min for temporal comparisons, but some values are invalid. Why and how many?
# determine how many prebd_min values are invalid
import os
import arcpy

# Set dataset path relative to working GDB
detections = os.path.join(working_gdb, "merged_94_24_nlcd_size_filtered")

total = 0
valid = 0

zero_vals = 0
invalid_month = 0
invalid_day = 0
malformed = 0

with arcpy.da.SearchCursor(detections, ["prebd_min"]) as cur:
    for (prebd,) in cur:
        total += 1

        # Case 1: zero or null
        if prebd in (0, None):
            zero_vals += 1
            continue

        # Convert float → int safely
        try:
            val = int(prebd)
        except Exception:
            malformed += 1
            continue

        s = str(val)

        # Must be exactly 8 digits (YYYYMMDD)
        if len(s) != 8:
            malformed += 1
            continue

        year = int(s[0:4])
        month = int(s[4:6])
        day = int(s[6:8])

        # Validate month/day ranges
        if not (1 <= month <= 12):
            invalid_month += 1
            continue

        if not (1 <= day <= 31):
            invalid_day += 1
            continue

        # If all checks passed
        valid += 1

# Summary
print("Total detections:", total)
print("Valid prebd_min:", valid)
print("Invalid total:", total - valid)

print("\nBreakdown of invalid values:")
print("  Zero values:", zero_vals)
print("  Invalid month:", invalid_month)
print("  Invalid day:", invalid_day)
print("  Malformed:", malformed)

def pct(x):
    return round((x / total) * 100, 2)

print("\nPercentages:")
print("  Zero values:", pct(zero_vals), "%")
print("  Invalid month:", pct(invalid_month), "%")
print("  Invalid day:", pct(invalid_day), "%")
print("  Malformed:", pct(malformed), "%")
print("  Valid:", pct(valid), "%")

In [ ]:
#fix prebd_min to prebd_min_corrected, fixes invalid days, fixes 0 to YYYY0101
import arcpy
import os

arcpy.env.overwriteOutput = True

fc = os.path.join(working_gdb, "merged_94_24_nlcd_size_filtered")

# Add corrected field if needed
fields = [f.name for f in arcpy.ListFields(fc)]
if "prebd_min_corrected" not in fields:
    arcpy.management.AddField(fc, "prebd_min_corrected", "LONG")

with arcpy.da.UpdateCursor(fc, ["prebd_min", "year", "prebd_min_corrected"]) as cur:
    for prebd, yr, corrected in cur:

        # Convert year to int safely
        yr_int = int(yr) if yr is not None else None

        # Case 1: zero or NULL → use YYYY0101
        if prebd in (0, None):
            fixed = int(f"{yr_int:04d}0101")
            cur.updateRow([prebd, yr, fixed])
            continue

        # Try converting float → int
        try:
            val = int(prebd)
        except Exception:
            fixed = int(f"{yr_int:04d}0101")
            cur.updateRow([prebd, yr, fixed])
            continue

        s = str(val)

        # Must be 8 digits
        if len(s) != 8:
            fixed = int(f"{yr_int:04d}0101")
            cur.updateRow([prebd, yr, fixed])
            continue

        # Extract components
        year = int(s[0:4])
        month = int(s[4:6])
        day = int(s[6:8])

        # Validate month
        if not (1 <= month <= 12):
            fixed = int(f"{yr_int:04d}0101")
            cur.updateRow([prebd, yr, fixed])
            continue

        # Clamp day into 1–31
        if day < 1:
            day = 1
        elif day > 31:
            day = 31

        # Rebuild corrected YYYYMMDD
        fixed = int(f"{year:04d}{month:02d}{day:02d}")

        cur.updateRow([prebd, yr, fixed])

print("Finished writing prebd_min_corrected.")

In [ ]:
#we want to use bd_min for temporal comparisons, but some values are invalid. Why and how many?
# check validity of bd_min values
import arcpy

detections = os.path.join(working_gdb, "merged_94_24_nlcd_size_filtered")

total = 0
valid = 0

zero_vals = 0
invalid_month = 0
invalid_day = 0
malformed = 0

with arcpy.da.SearchCursor(detections, ["bd_min"]) as cur:
    for (bd,) in cur:
        total += 1

        # Case 1: zero or null
        if bd in (0, None):
            zero_vals += 1
            continue

        # Convert float → int
        try:
            val = int(bd)
        except Exception:
            malformed += 1
            continue

        s = str(val)

        # Must be 8 digits
        if len(s) != 8:
            malformed += 1
            continue

        year = int(s[0:4])
        month = int(s[4:6])
        day = int(s[6:8])

        # Check month/day
        if not (1 <= month <= 12):
            invalid_month += 1
            continue

        if not (1 <= day <= 31):
            invalid_day += 1
            continue

        valid += 1

# Summary
print("Total detections:", total)
print("Valid bd_min:", valid)
print("Invalid total:", total - valid)

print("\nBreakdown of invalid values:")
print("  Zero values:", zero_vals)
print("  Invalid month:", invalid_month)
print("  Invalid day:", invalid_day)
print("  Malformed:", malformed)

def pct(x):
    return round((x / total) * 100, 2)

print("\nPercentages:")
print("  Zero values:", pct(zero_vals), "%")
print("  Invalid month:", pct(invalid_month), "%")
print("  Invalid day:", pct(invalid_day), "%")
print("  Malformed:", pct(malformed), "%")
print("  Valid:", pct(valid), "%")

In [ ]:
#Now correct bd_min to new field bd_min_corrected
import arcpy

arcpy.env.overwriteOutput = True

fc = os.path.join(working_gdb, "merged_94_24_nlcd_size_filtered")

# Add corrected field if needed
fields = [f.name for f in arcpy.ListFields(fc)]
if "bd_min_corrected" not in fields:
    arcpy.management.AddField(fc, "bd_min_corrected", "LONG")

with arcpy.da.UpdateCursor(fc, ["bd_min", "year", "bd_min_corrected"]) as cur:
    for bd, yr, corrected in cur:

        # Convert year (float) → int
        yr_int = int(yr)

        # Case 1: zero or NULL → fallback to YYYY1231
        if bd in (0, None):
            fixed = int(f"{yr_int:04d}1231")
            cur.updateRow([bd, yr, fixed])
            continue

        # Try converting float → int
        try:
            val = int(bd)
        except Exception:
            fixed = int(f"{yr_int:04d}1231")
            cur.updateRow([bd, yr, fixed])
            continue

        s = str(val)

        # Must be 8 digits
        if len(s) != 8:
            fixed = int(f"{yr_int:04d}1231")
            cur.updateRow([bd, yr, fixed])
            continue

        # Extract components
        year = int(s[0:4])
        month = int(s[4:6])
        day = int(s[6:8])

        # Validate month
        if not (1 <= month <= 12):
            fixed = int(f"{yr_int:04d}1231")
            cur.updateRow([bd, yr, fixed])
            continue

        # Clamp day into 1–31
        if day < 1:
            day = 1
        elif day > 31:
            day = 31

        # Rebuild corrected YYYYMMDD
        fixed = int(f"{year:04d}{month:02d}{day:02d}")

        cur.updateRow([bd, yr, fixed])

print("Finished writing bd_min_corrected.")

In [ ]:
#Add 8 days to bd_min_corrected for end of time interval: 
import arcpy
from datetime import datetime, timedelta

arcpy.env.overwriteOutput = True

fc = os.path.join(working_gdb, "merged_94_24_nlcd_size_filtered")

# Add field if missing
fields = [f.name for f in arcpy.ListFields(fc)]
if "bd_min_corrected_plus8" not in fields:
    arcpy.management.AddField(fc, "bd_min_corrected_plus8", "LONG")

with arcpy.da.UpdateCursor(fc, ["bd_min_corrected", "bd_min_corrected_plus8"]) as cur:
    for bd_corr, plus8 in cur:

        # Convert YYYYMMDD → datetime
        s = str(int(bd_corr))
        year  = int(s[0:4])
        month = int(s[4:6])
        day   = int(s[6:8])

        date_obj = datetime(year, month, day)

        # Add 8 days
        new_date = date_obj + timedelta(days=8)

        # Convert back to YYYYMMDD integer
        new_val = int(new_date.strftime("%Y%m%d"))

        cur.updateRow([bd_corr, new_val])

print("bd_min_corrected_plus8 field populated.")